In [2]:
try:
    !pip install -q \
    ragas \
    datasets \
    chromadb \
    langchain \
    langchain-community \
    langchain-google-genai \
    rank-bm25 \
    groq

    print(" Libraries installed")

except Exception as e:
    print(f" Installation Error: {e}")

 Libraries installed


In [3]:
try:
    from google.colab import drive

    drive.mount('/content/drive')

    print("✅ Drive mounted")

except Exception as e:
    print(f"❌ Drive Mount Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted


In [4]:
try:

    import os
    import ast

    from datasets import Dataset

    from google.colab import userdata

    from groq import Groq

    from rank_bm25 import BM25Okapi

    from langchain_community.vectorstores import Chroma

    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    print(" Imports successful")

except Exception as e:
    print(f" Import Error: {e}")

 Imports successful


In [5]:
try:
    GEMINI_API_KEY = userdata.get(
        "Gemini_key"
    )
    GROQ_API_KEY = userdata.get(
        "Groq_Key"
    )
    if not GEMINI_API_KEY:
        raise ValueError(
            "Gemini API key missing"
        )
    if not GROQ_API_KEY:
        raise ValueError(
            "Groq API key missing"
        )
    print(" API Keys Loaded")

except Exception as e:
    print(f" API Key Error: {e}")

 API Keys Loaded


In [6]:
try:
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GEMINI_API_KEY
    )
    print(" Embeddings initialized")

except Exception as e:
    print(f" Embedding Error: {e}")

 Embeddings initialized


In [7]:
try:
    CHROMA_PATH = (
        "/content/drive/MyDrive/RAG_Internship/phase4_chroma_db"
    )
    vectordb = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embeddings,
        collection_name="wiki_rag_phase4"
    )
    print(" Phase 4 ChromaDB Loaded")

except Exception as e:

    print(f" Chroma Load Error: {e}")

/tmp/ipykernel_31322/576483151.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(


 Phase 4 ChromaDB Loaded


In [8]:
try:

    docs = vectordb.get()
    texts = docs["documents"]
    tokenized_corpus = [
        text.split()
        for text in texts
    ]
    bm25 = BM25Okapi(
        tokenized_corpus
    )
    print(
        f" BM25 Rebuilt"
    )
    print(
        f"Documents Loaded: {len(texts)}"
    )

except Exception as e:
    print(f" BM25 Error: {e}")

 BM25 Rebuilt
Documents Loaded: 347


Hybrid Search Function

In [9]:
try:

    from langchain_core.documents import Document
    def hybrid_search(
        query,
        k=5
    ):
        vector_results = (
            vectordb.similarity_search(
                query,
                k=k
            )
        )
        tokenized_query = (
            query.split()
        )
        bm25_scores = (
            bm25.get_scores(
                tokenized_query
            )
        )
        top_indices = (
            sorted(
                range(
                    len(bm25_scores)
                ),
                key=lambda i:
                bm25_scores[i],
                reverse=True
            )[:k]
        )
        bm25_docs = [
            Document(
                page_content=texts[i]
            )
            for i in top_indices
        ]
        combined = (
            vector_results +
            bm25_docs
        )
        unique_docs = []
        seen = set()

        for doc in combined:
            if (
                doc.page_content
                not in seen
            ):
                seen.add(
                    doc.page_content
                )
                unique_docs.append(
                    doc
                )
        return unique_docs[:k]
    print(" Hybrid Search Ready")

except Exception as e:
    print(f" Hybrid Search Error: {e}")

 Hybrid Search Ready


In [10]:
try:
    docs = hybrid_search(
        "What is machine learning?"
    )
    print(
        f"Retrieved {len(docs)} documents"
    )
    print("\nFirst document:\n")
    print(
        docs[0].page_content[:500]
    )
except Exception as e:
    print(
        f"Test Error: {e}"
    )

Retrieved 5 documents

First document:

Models
A machine learning model is a type of mathematical model that, once "trained" on a given dataset, can be used to make predictions or classifications on new data. During training, a learning algorithm iteratively adjusts the model's internal parameters to minimise errors in its predictions. By extension, the term "model" can refer to several levels of specificity, from a general class of models and their associated learning algorithms to a fully trained model with all its internal paramete


Answer Generation

In [11]:
try:
    client = Groq(
        api_key=GROQ_API_KEY
    )
    print(
        " Groq Client Ready"
    )
except Exception as e:
    print(f" Groq Error: {e}")

 Groq Client Ready


In [12]:
try:

    def ask_question(query):

        try:

            docs = hybrid_search(
                query
            )

            if len(docs) == 0:

                return (
                    "No relevant documents found."
                )

            context = "\n\n".join(
                [
                    doc.page_content
                    for doc in docs
                ]
            )

            prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not present in the context say:

I could not find that information in the documents.

Context:
{context}

Question:
{query}
"""

            response = (
                client.chat.completions.create(
                    model=
                    "llama-3.3-70b-versatile",
                    messages=[
                        {
                            "role":"user",
                            "content":prompt
                        }
                    ]
                )
            )

            return (
                response
                .choices[0]
                .message
                .content
            )

        except Exception as e:

            return f"Error: {e}"

    print(
        " ask_question Ready"
    )

except Exception as e:
    print(f" Function Error: {e}")

 ask_question Ready


In [13]:
try:

    with open(
        "/content/arrays.txt",
        "r",
        encoding="utf-8"
    ) as f:
        data = f.read()
    namespace = {}
    exec(
        data,
        namespace
    )
    questions = (
        namespace["test_questions"]
    )
    ground_truths = (
        namespace["ground_truths"]
    )
    print(
        f"Questions: {len(questions)}"
    )
    print(
        f"Ground Truths: {len(ground_truths)}"
    )

except Exception as e:
    print(
        f" Array Load Error: {e}"
    )

Questions: 20
Ground Truths: 20


Testing for 2 Questions

In [14]:
try:

    questions = questions[:2]
    ground_truths = (
        ground_truths[:2]
    )
    print(
        " Using first 2 questions"
    )

except Exception as e:
    print(f" Error: {e}")

 Using first 2 questions


In [15]:
try:
    answers = []
    contexts = []
    for i, question in enumerate(
        questions,
        start=1
    ):
        print(
            f"Processing {i}"
        )
        docs = hybrid_search(
            question
        )
        contexts.append(
            [
                doc.page_content
                for doc in docs
            ]
        )
        answer = ask_question(
            question
        )
        answers.append(
            answer
        )
    print(
        " Answers Generated"
    )

except Exception as e:
    print(f" Generation Error: {e}")

Processing 1
Processing 2
 Answers Generated


create Dataset for RAGAS

In [16]:
try:

    dataset = Dataset.from_dict(
        {
            "question":
            questions,

            "answer":
            answers,

            "contexts":
            contexts,

            "ground_truth":
            ground_truths
        }
    )

    print(
        " Dataset Created"
    )

except Exception as e:
    print(f" Dataset Error: {e}")

 Dataset Created


RAGAS Evaluation

In [17]:
print(type(dataset))
print(type(GROQ_API_KEY))
print(type(GEMINI_API_KEY))

<class 'datasets.arrow_dataset.Dataset'>
<class 'str'>
<class 'str'>


In [ ]:
try:

    from langchain_groq import ChatGroq
    from langchain_google_genai import (
        GoogleGenerativeAIEmbeddings
    )

    evaluator_llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_API_KEY,
        temperature=0
    )

    evaluator_embeddings = (
        GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=GEMINI_API_KEY
        )
    )

    result = evaluate(
        dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )
    print("\nRAGAS Results")
    print("-" * 50)
    print(result)
    print("\nMetric Values")
    print("-" * 50)
    result_dict = result
    if hasattr(result, "to_dict"):
        result_dict = result.to_dict()
    for metric, value in result_dict.items():

        try:
            print(
                f"{metric}: "
                f"{value:.4f}"
            )
        except:
            print(
                f"{metric}: {value}"
            )

except Exception as e:
    print(
        f" RAGAS Error: {e}"
    )

In [21]:
print(type(result))
print(result)

<class 'ragas.dataset_schema.EvaluationResult'>
{'faithfulness': 1.0000, 'answer_relevancy': 1.0000, 'context_precision': 0.9333, 'context_recall': 1.0000}


In [22]:
try:
    df = (
        result.to_pandas()
    )

    display(df)

except Exception as e:
    print(
        f" DataFrame Error: {e}"
    )

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is machine learning?,[Models\nA machine learning model is a type of...,Machine learning is the study of programs that...,Machine learning is a field of study in artifi...,NaN,1.0,0.866667,1.0
1,What is deep learning?,[Overview\nMost modern deep learning models ar...,Deep learning refers to a class of machine lea...,Deep learning refers to a class of machine lea...,1.0,1.0,1.000000,1.0


In [23]:
try:

    save_path = (
        "/content/drive/MyDrive/"
        "Phase4_RAGAS_2Q.csv"
    )

    df.to_csv(
        save_path,
        index=False
    )

    print(
        f" Saved: {save_path}"
    )

except Exception as e:
    print(
        f" Save Error: {e}"
    )

 Saved: /content/drive/MyDrive/Phase4_RAGAS_2Q.csv


# Phase 4 RAGAS Evaluation Summary

## Objective
The objective of this notebook is to evaluate the performance of the Phase 4 Retrieval-Augmented Generation (RAG) system using the RAGAS evaluation framework.

Phase 4 enhances the baseline RAG pipeline by introducing **Hybrid Retrieval**, which combines:

- Semantic Vector Search (ChromaDB)
- BM25 Keyword Search

This approach aims to improve retrieval quality and answer relevance.

---

## Evaluation Workflow

1. Load the existing ChromaDB knowledge base.
2. Rebuild the BM25 index from stored documents.
3. Perform Hybrid Retrieval for each query.
4. Generate answers using the configured LLM.
5. Create an evaluation dataset consisting of:
   - Question
   - Generated Answer
   - Retrieved Contexts
   - Ground Truth Answer
6. Evaluate the system using RAGAS metrics.
7. Save the evaluation results as a CSV file.

---

## Evaluation Metrics

The following RAGAS metrics were used:

### Faithfulness
Measures whether the generated answer is supported by the retrieved context.

### Answer Relevancy
Measures how relevant the generated answer is to the user's question.

### Context Precision
Measures how much of the retrieved context is actually useful for answering the question.

### Context Recall
Measures whether the retrieval process successfully retrieves all relevant information needed to answer the question.

---

## Results (2-Question Validation Run)

| Metric | Score |
|----------|----------|
| Faithfulness | 1.0000 |
| Answer Relevancy | 1.0000 |
| Context Precision | 0.9333 |
| Context Recall | 1.0000 |

---

## Comparison with Phase 2

| Metric | Phase 2 | Phase 4 |
|----------|----------|----------|
| Faithfulness | 1.0000 | 1.0000 |
| Answer Relevancy | 0.7897 | 1.0000 |
| Context Precision | 1.0000 | 0.9333 |
| Context Recall | 1.0000 | 1.0000 |

---

## Observations

- Hybrid Retrieval successfully improved answer relevancy.
- Faithfulness remained perfect, indicating answers are grounded in retrieved documents.
- Context recall remained perfect, showing that relevant information was successfully retrieved.
- Context precision decreased slightly due to additional documents retrieved through BM25.
- Overall, Phase 4 produced more relevant answers while maintaining strong retrieval quality.

---

## Conclusion

The Phase 4 Hybrid RAG system was successfully implemented and evaluated using the RAGAS framework. The results demonstrate improved answer quality compared to the baseline Phase 2 implementation while maintaining high faithfulness and retrieval effectiveness.

